In [1]:
"""
RSNA Knee Abnormality Detection — Starter Training Pipeline
Targets: ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA,
         PF OA, Effusion, Synovitis, Baker's, Contusion, Fracture
Metric:  Macro-averaged AUC-ROC
"""

import os
import random
import warnings
from pathlib import Path

import albumentations as A
import numpy as np
import pandas as pd
import pydicom
import timm
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

warnings.filterwarnings("ignore")

from pathlib import Path

PREBUILT_CACHE_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection/dicom_cache")  # for future sessions, once you've built and uploaded one -- doesn't exist yet, that's fine
LOCAL_CACHE_DIR = Path("/kaggle/working/dicom_cache")
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
# ── Config ─────────────────────────────────────────────────────────────────────

class CFG:
    # Paths — update DATA_DIR for local use; /kaggle/input/... on Kaggle
    DATA_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
    OUTPUT_DIR = Path("/kaggle/working")

    TARGETS = [
        "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
        "Medial OA", "Lateral OA", "PF OA", "Effusion",
        "Synovitis", "Baker's", "Contusion", "Fracture",
    ]
    NUM_CLASSES = len(TARGETS)  # 12

    # Slices uniformly sampled per series
    N_SLICES = 12

    # Model — MERGED: now DINOv2 + attention pooling (RSNAModelV2), not EfficientNet-B3.
    # The original "efficientnet_b3" name is kept below, commented, purely as a fast revert path.
    #MODEL_NAME = "vit_small_patch14_dinov2.lvd142m"
    MODEL_NAME = "efficientnet_b3"  # <- old backbone, revert to this + use RSNAModel() in main() if needed
    IMG_SIZE = 224

    # Training
    EPOCHS = 2
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 4  # effective batch size = BATCH_SIZE * GRAD_ACCUM_STEPS
    ENCODER_CHUNK_SIZE = 16  # max slices sent through the encoder at once (caps activation memory)
    LR = 1e-4
    WEIGHT_DECAY = 1e-2
    FOLDS = 2
    SEED = 42

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    NUM_WORKERS = 4


# ── DICOM Utilities ────────────────────────────────────────────────────────────

def load_dicom_volume(series_dir: Path) -> np.ndarray:
    """Load all slices in a series; sort by InstanceNumber when available."""
    dcm_files = list(series_dir.glob("*.dcm"))

    def sort_key(f):
        try:
            return int(pydicom.dcmread(str(f), stop_before_pixels=True).InstanceNumber)
        except Exception:
            return f.name

    dcm_files.sort(key=sort_key)

    slices = []
    for f in dcm_files:
        dcm = pydicom.dcmread(str(f))
        arr = dcm.pixel_array.astype(np.float32)
        slope = float(getattr(dcm, "RescaleSlope", 1))
        intercept = float(getattr(dcm, "RescaleIntercept", 0))
        slices.append(arr * slope + intercept)

    return np.stack(slices, axis=0)  # (D, H, W)


def normalize_volume(volume: np.ndarray) -> np.ndarray:
    """Clip to 1st/99th percentile then scale to [0, 1]."""
    p1, p99 = np.percentile(volume, [1, 99])
    volume = np.clip(volume, p1, p99)
    volume = (volume - p1) / (p99 - p1 + 1e-6)
    return volume.astype(np.float32)


def sample_slices(volume: np.ndarray, n: int) -> np.ndarray:
    """Uniformly sample n slices along the depth axis."""
    indices = np.linspace(0, volume.shape[0] - 1, n, dtype=int)
    return volume[indices]  # (n, H, W)


def pick_best_series(study_uid: str, series_df: pd.DataFrame) -> str:
    """Prefer sagittal fluid-sensitive series; fall back to first available."""
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    preferred = rows[
        (rows["Anatomical_Plane"] == "Sagittal") & (rows["Fluid_Sensitive"] == 1)
    ]
    chosen = preferred if len(preferred) > 0 else rows
    return chosen.iloc[0]["SeriesInstanceUID"]

def get_prioritized_series(study_uid: str, series_df: pd.DataFrame, max_series: int = 3) -> list[str]:
    """
    Returns up to max_series series UIDs for this study, ordered by priority --
    sagittal fluid-sensitive first (reusing your original pick_best_series
    logic), then everything else. Real studies vary in how many series they
    actually have; this handles that gracefully rather than assuming a fixed count.
    """
    rows = series_df[series_df["StudyInstanceUID"] == study_uid]
    preferred = rows[(rows["Anatomical_Plane"] == "Sagittal") & (rows["Fluid_Sensitive"] == 1)]
    others = rows[~rows.index.isin(preferred.index)]
    ordered = pd.concat([preferred, others])
    return ordered["SeriesInstanceUID"].tolist()[:max_series]
# ── Dataset ────────────────────────────────────────────────────────────────────

class RSNADataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        series_df: pd.DataFrame,
        split: str = "train",    # "train" | "test"
        transform=None,
    ):
        self.df = df.reset_index(drop=True)
        self.series_df = series_df
        self.split = split
        self.transform = transform
        self.series_root = CFG.DATA_DIR / f"{split}_series"

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        import cv2
        row = self.df.iloc[idx]
        study_uid = row["StudyInstanceUID"]

        prebuilt_path = PREBUILT_CACHE_DIR / f"{study_uid}.npy"
        local_path = LOCAL_CACHE_DIR / f"{study_uid}.npy"

        if prebuilt_path.exists():
            all_slices = np.load(prebuilt_path).astype(np.float32)
        elif local_path.exists():
            all_slices = np.load(local_path).astype(np.float32)
        else:
            series_uids = get_prioritized_series(study_uid, self.series_df)
            n_series = len(series_uids)
            slices_per_series = CFG.N_SLICES // n_series
            remainder = CFG.N_SLICES % n_series

            all_slices_list = []
            for i, series_uid in enumerate(series_uids):
                series_dir = self.series_root / study_uid / series_uid
                volume = load_dicom_volume(series_dir)
                volume = normalize_volume(volume)
                n_this_series = slices_per_series + (1 if i < remainder else 0)
                slices = sample_slices(volume, n_this_series)
                resized_slices = np.stack([
                cv2.resize(s, (CFG.IMG_SIZE, CFG.IMG_SIZE)) for s in slices
                ])
                all_slices_list.append(resized_slices)

            all_slices = np.concatenate(all_slices_list, axis=0)
            np.save(local_path, all_slices.astype(np.float16))

        images = []
        for s in all_slices:
            img = np.stack([s, s, s], axis=-1)
            if self.transform:
                img = self.transform(image=img)["image"]
            images.append(img)
        images = torch.stack(images, dim=0)
        if self.split == "train":
            raw_labels = row[CFG.TARGETS].values.astype(np.float32)
            valid_mask = (~pd.isnull(raw_labels)).astype(np.float32)
            labels = np.nan_to_num(raw_labels, nan=0.0)

            labels = torch.tensor(labels, dtype=torch.float32)
            valid_mask = torch.tensor(valid_mask, dtype=torch.float32)
            weight = torch.tensor(row["sample_weight"], dtype=torch.float32)
            return images, labels, valid_mask, weight

        return images, study_uid


# ── Transforms ─────────────────────────────────────────────────────────────────

def get_transforms(is_train: bool) -> A.Compose:
    if is_train:
        return A.Compose([
            A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
            A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
            ToTensorV2(),
        ])
    return A.Compose([
        A.Resize(CFG.IMG_SIZE, CFG.IMG_SIZE),
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ToTensorV2(),
    ])


# ── Model (OLD) — kept, unused, as a fast revert path ───────────────────────────

class RSNAModel(nn.Module):
    """
    2.5D model: encode each slice independently with a CNN backbone,
    then mean-pool slice features before the classification head.
    NOT USED after the merge below — kept only so you can revert quickly
    by changing MODEL_NAME back and swapping RSNAModelV2() -> RSNAModel() in main().
    """

    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=0, global_pool="avg"
        )
        if hasattr(self.encoder, "set_grad_checkpointing"):
            self.encoder.set_grad_checkpointing(True)
        feat_dim = self.encoder.num_features
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CFG.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C, H, W = x.shape
        flat = x.view(B * N, C, H, W)
        feats = torch.cat(
            [self.encoder(chunk) for chunk in flat.split(CFG.ENCODER_CHUNK_SIZE, dim=0)],
            dim=0,
        )
        feats = feats.view(B, N, -1).mean(dim=1)       # mean over slices → (B, feat_dim)
        return self.head(feats)                         # (B, num_classes)


# ── Model (NEW) — MERGED from rsna_model_v2_attention.py ───────────────────────

class AttentionPool(nn.Module):
    """
    Takes N slice-level feature vectors, learns which ones matter most for
    THIS study, and returns one weighted-combination feature vector.
    Verified correct (numpy cross-check): softmax over the slice dimension,
    weights sum to 1 per study, genuine weighted combination confirmed.
    """

    def __init__(self, feat_dim: int, hidden_dim: int = 128):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Linear(feat_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, slice_features: torch.Tensor) -> torch.Tensor:
        """slice_features: (B, N, feat_dim). Returns: (B, feat_dim)."""
        raw_scores = self.attention(slice_features)  # (B, N, 1)
        scores = raw_scores.squeeze(-1)               # (B, N)
        weights = torch.softmax(scores, dim=1)         # softmax across slices, per study
        pooled = torch.bmm(weights.unsqueeze(1), slice_features).squeeze(1)  # (B, feat_dim)
        return pooled


class RSNAModelV2(nn.Module):
    """
    RSNA Knee model with DINOv2 backbone and attention-based slice pooling.
    This is now what main() actually trains.
    """

    def __init__(self):
        super().__init__()
        self.encoder = timm.create_model(
            CFG.MODEL_NAME, pretrained=True, num_classes=0, img_size=CFG.IMG_SIZE
        )
        feat_dim = self.encoder.num_features
        self.pool = AttentionPool(feat_dim)
        self.head = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, CFG.NUM_CLASSES),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C, H, W = x.shape
        flat = x.view(B * N, C, H, W)
        feats = torch.cat(
            [self.encoder(chunk) for chunk in flat.split(CFG.ENCODER_CHUNK_SIZE, dim=0)],
            dim=0,
        )
        feats = feats.view(B, N, -1)          # (B, N, feat_dim) — NOT pooled yet
        pooled = self.pool(feats)               # attention pooling replaces .mean(dim=1)
        return self.head(pooled)                # (B, num_classes)


# ── Metrics ────────────────────────────────────────────────────────────────────

def macro_auc(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Macro AUC-ROC; skips any target with only one unique class in the batch."""
    aucs = []
    for i in range(y_true.shape[1]):
        if len(np.unique(y_true[:, i])) > 1:
            aucs.append(roc_auc_score(y_true[:, i], y_pred[:, i]))
    return float(np.mean(aucs)) if aucs else 0.0


# ── Train / Validate ───────────────────────────────────────────────────────────

def train_one_epoch(model, loader, optimizer, criterion, scaler, device):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)
    for step, (images, labels, valid_mask, weight) in enumerate(loader):
        images, labels, valid_mask, weight = images.to(device), labels.to(device), valid_mask.to(device), weight.to(device)
        logits = model(images)
        if step == 0:
            with torch.no_grad():
                probs = torch.sigmoid(logits)
                print(f"Prediction std across batch: {probs.std(dim=0)}")
                print(f"Min: {probs.min(dim=0).values}, Max: {probs.max(dim=0).values}")
        raw_loss = criterion(logits, labels)              # shape (batch, 12)
        raw_loss = raw_loss * valid_mask                   # zero out anything that was null
        raw_loss = raw_loss * weight.unsqueeze(1)          # scale each sample by its source confidence

        loss = raw_loss.sum() / valid_mask.sum().clamp(min=1) / CFG.GRAD_ACCUM_STEPS
        loss.backward()
        
        if step == 0:
            encoder_grad_norm = 0.0
            head_grad_norm = 0.0
            for name, param in model.named_parameters():
                if param.grad is not None:
                    if 'encoder' in name:
                        encoder_grad_norm += param.grad.norm().item()
                    else:
                        head_grad_norm += param.grad.norm().item()
            print(f"Encoder gradient norm: {encoder_grad_norm:.4f}")
            print(f"Head+pool gradient norm: {head_grad_norm:.4f}")
        if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        total_loss += loss.item() * CFG.GRAD_ACCUM_STEPS
        if step == 0:
            print("valid_mask sum:", valid_mask.sum().item(), "of", valid_mask.numel())
            print("raw_loss sum before mask:", raw_loss.sum().item())
            print("raw_loss sum after mask+weight:", (raw_loss * valid_mask * weight.unsqueeze(1)).sum().item())
    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, labels, valid_mask, weight in loader:
        images, labels, valid_mask, weight = images.to(device), labels.to(device), valid_mask.to(device), weight.to(device)
        with torch.amp.autocast("cuda"):
            logits = model(images)
            raw_loss = criterion(logits, labels)
            raw_loss = raw_loss * valid_mask
            batch_loss = raw_loss.sum() / valid_mask.sum().clamp(min=1)
            total_loss += batch_loss.item()
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    preds = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)
    return total_loss / len(loader), macro_auc(labels, preds)


# ── Main ───────────────────────────────────────────────────────────────────────

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def main():
    seed_everything(CFG.SEED)
    CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    train_df = pd.read_csv(CFG.DATA_DIR / "train.csv")
    series_df = pd.read_csv(CFG.DATA_DIR / "train_series.csv")
    hard_labeled_df = train_df.dropna(subset=CFG.TARGETS).copy()
    for col in CFG.TARGETS:
        hard_labeled_df[col] = hard_labeled_df[col].astype(float)
    hard_labeled_df["sample_weight"] = 1.0

    import json
    with open("/kaggle/input/datasets/chiragggg/extracted-labels-llama/extracted_labels.json", "r") as f:
        extracted = json.load(f)

    extracted_df = pd.DataFrame.from_dict(extracted, orient="index")
    extracted_df.index.name = "StudyInstanceUID"
    extracted_df = extracted_df.reset_index()
    extracted_df["sample_weight"] = 0.7
    extracted_df = extracted_df[~extracted_df["StudyInstanceUID"].isin(hard_labeled_df["StudyInstanceUID"])]

    labeled_df = pd.concat([hard_labeled_df, extracted_df], ignore_index=True)

    print(f"Hard-labeled: {len(hard_labeled_df)}, Extracted: {len(extracted_df)}, Total: {len(labeled_df)}")
    print(f"Labeled studies : {len(labeled_df)}")
    print(f"Label prevalence:\n{labeled_df[CFG.TARGETS].mean().round(3).to_string()}\n")

    skf = StratifiedKFold(n_splits=CFG.FOLDS, shuffle=True, random_state=CFG.SEED)
    hard_labeled_df["fold"] = -1
    for fold, (_, val_idx) in enumerate(skf.split(hard_labeled_df, hard_labeled_df["ACL"])):
        hard_labeled_df.loc[hard_labeled_df.index[val_idx], "fold"] = fold

    device = torch.device(CFG.DEVICE)
    criterion = nn.BCEWithLogitsLoss(reduction='none')

    for fold in range(CFG.FOLDS):
        print(f"\n{'─'*45}")
        print(f"  Fold {fold + 1}/{CFG.FOLDS}")
        print(f"{'─'*45}")

        train_fold = pd.concat([hard_labeled_df[hard_labeled_df["fold"] != fold], extracted_df], ignore_index=True)
        val_fold = hard_labeled_df[hard_labeled_df["fold"] == fold]

        train_loader = DataLoader(
            RSNADataset(train_fold, series_df, "train", get_transforms(True)),
            batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=CFG.NUM_WORKERS, pin_memory=True,
        )
        val_loader = DataLoader(
            RSNADataset(val_fold, series_df, "train", get_transforms(False)),
            batch_size=CFG.BATCH_SIZE, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True,
        )

        model = RSNAModel().to(device)  # MERGED: was RSNAModel()
        optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY
        )
        #optimizer = torch.optim.AdamW([
    #{"params": model.encoder.parameters(), "lr": CFG.LR * 0.1},  # the experienced radiologist -- gentle updates
    #{"params": model.pool.parameters(), "lr": CFG.LR},             # the intern -- needs to learn fast
    #{"params": model.head.parameters(), "lr": CFG.LR},             # same -- starting from zero
#], weight_decay=CFG.WEIGHT_DECAY)
        
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=CFG.EPOCHS
        )
        scaler = torch.amp.GradScaler()

        best_auc = 0.0
        for epoch in range(CFG.EPOCHS):
            tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler, device)
            val_loss, val_auc = validate(model, val_loader, criterion, device)
            scheduler.step()

            flag = "  ←" if val_auc > best_auc else ""
            print(
                f"  Epoch {epoch+1:02d}  "
                f"train_loss={tr_loss:.4f}  "
                f"val_loss={val_loss:.4f}  "
                f"val_auc={val_auc:.4f}{flag}"
            )

            if val_auc > best_auc:
                best_auc = val_auc
                torch.save(
                    model.state_dict(),
                    CFG.OUTPUT_DIR / f"fold{fold}_best.pth",
                )

        print(f"\n  Best AUC fold {fold+1}: {best_auc:.4f}")
        del model, optimizer, scheduler, scaler
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()

Hard-labeled: 58, Extracted: 4349, Total: 4407
Labeled studies : 4407
Label prevalence:
ACL                 0.294
MCL                 0.181
Medial Meniscus     0.536
Lateral Meniscus    0.242
Medial OA           0.567
Lateral OA          0.261
PF OA               0.460
Effusion            0.642
Synovitis           0.369
Baker's             0.415
Contusion           0.128
Fracture            0.070


─────────────────────────────────────────────
  Fold 1/2
─────────────────────────────────────────────


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Prediction std across batch: tensor([0.0062, 0.0035, 0.0067, 0.0063, 0.0057, 0.0045, 0.0016, 0.0027, 0.0063,
        0.0093, 0.0053, 0.0042], device='cuda:0')
Min: tensor([0.4892, 0.5056, 0.4972, 0.4909, 0.5077, 0.4842, 0.4930, 0.4852, 0.5157,
        0.4843, 0.5224, 0.5067], device='cuda:0'), Max: tensor([0.5032, 0.5140, 0.5132, 0.5046, 0.5188, 0.4948, 0.4967, 0.4915, 0.5310,
        0.5047, 0.5339, 0.5164], device='cuda:0')
Encoder gradient norm: 0.2047
Head+pool gradient norm: 0.0864
valid_mask sum: 45.0 of 48
raw_loss sum before mask: 21.974021911621094
raw_loss sum after mask+weight: 15.381814956665039
  Epoch 01  train_loss=0.4101  val_loss=0.6731  val_auc=0.5857  ←
Prediction std across batch: tensor([0.1026, 0.0687, 0.0609, 0.0514, 0.0497, 0.0824, 0.0523, 0.0421, 0.0865,
        0.0877, 0.0714, 0.0530], device='cuda:0')
Min: tensor([0.2315, 0.1024, 0.4542, 0.2125, 0.5331, 0.1613, 0.4339, 0.6213, 0.2715,
        0.2280, 0.0542, 0.0291], device='cuda:0'), Max: tensor([0.4126, 0.2